In [2]:
from ultralytics import YOLO
import cv2
import os
import math

In [3]:
#LOAD THE EXISTING YOLO model
model = YOLO("yolov8n.pt")

print("YOLO model loaded successfully")

YOLO model loaded successfully


In [4]:
video_path = "Trafficvedio.mp4"

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

duration = frame_count / fps


print("FPS:", fps)
print("Frames:", frame_count)
print("Resolution:", width, "x", height)
print("Duration:", round(duration, 2), "seconds")

cap.release()

FPS: -1.0
Frames: -1
Resolution: -1 x -1
Duration: 1.0 seconds


In [6]:
import os

print("Current folder:")
print(os.getcwd())

print("\nFiles in current folder:")
print(os.listdir())

Current folder:
C:\Users\DELL

Files in current folder:
['.antigravity-ide', '.arduinoIDE', '.copilot', '.gemini', '.gitconfig', '.ipynb_checkpoints', '.ipython', '.jupyter', '.python_history', '.vscode', '.vscode-shared', 'AppData', 'Application Data', 'Contacts', 'Cookies', 'CrossDevice', 'Documents', 'Downloads', 'ETS2 OpenGL.lnk', 'Favorites', 'FluxI64.exe', 'Links', 'Local Settings', 'My Documents', 'NetHood', 'NTUSER.DAT', 'ntuser.dat.LOG1', 'ntuser.dat.LOG2', 'NTUSER.DAT{efcf1a20-f05e-11ee-9a53-f423e5c88176}.TM.blf', 'NTUSER.DAT{efcf1a20-f05e-11ee-9a53-f423e5c88176}.TMContainer00000000000000000001.regtrans-ms', 'NTUSER.DAT{efcf1a20-f05e-11ee-9a53-f423e5c88176}.TMContainer00000000000000000002.regtrans-ms', 'ntuser.ini', 'OneDrive', 'OneDrive - Amrita Vishwa Vidyapeetham', 'PrintHood', 'Recent', 'Saved Games', 'Searches', 'SendTo', 'Start Menu', 'Templates', 'vehicle_speed_initial.ipynb', 'yolov8n.pt']


In [8]:
import os
import cv2

video_path = r"C:\Users\DELL\OneDrive\Desktop\Final\Trafficvedio.mp4"

print("Video path:")
print(video_path)

print("File exists:", os.path.exists(video_path))


Video path:
C:\Users\DELL\OneDrive\Desktop\Final\Trafficvedio.mp4
File exists: True


In [9]:
cap = cv2.VideoCapture(video_path)

print("Video opened:", cap.isOpened())

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

print("FPS:", fps)
print("Frames:", frame_count)
print("Resolution:", width, "x", height)

if fps > 0:
    print("Duration:", round(frame_count / fps, 2), "seconds")

cap.release()

Video opened: True
FPS: 23.976023976023978
Frames: 660
Resolution: 1920 x 1080
Duration: 27.53 seconds


In [10]:
results = model.track(
    source=video_path,
    save=True,
    show=False,
    persist=True,
    classes=[2, 3, 5, 7],
    verbose=True
)

print("Vehicle tracking completed")


WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/660) C:\Users\DELL\OneDrive\Desktop\Final\Trafficvedio.mp4: 384x640 9 cars, 1 truck, 598.2ms
video 1/1 (frame 2/660) C:\Users\DELL\OneDrive\Desktop\Final\Trafficvedio.mp4: 384x640 9 cars, 1 truck, 262.2ms
video 1/1 (frame 3/660) C:\Users\DELL\OneDrive\Desktop\Final\Trafficvedio.mp4: 384x640 9 cars, 1 truck, 304.8ms
video 1/1 (frame 4/660) C:\Users\DELL\OneDrive\Desktop\Final\Trafficvedio.mp4: 384x640 9 cars, 1 truck, 344.1ms
video 1/1 (fr

In [11]:
import os

output_folder = r"C:\Users\DELL\runs\detect\track"

print(os.listdir(output_folder))

['Trafficvedio.avi']


In [12]:
vehicle_count = 0

for result in results:
    if result.boxes is not None:
        vehicle_count += len(result.boxes)

print("Total detected vehicles:", vehicle_count)

Total detected vehicles: 10481


In [13]:
vehicle_ids = set()

for result in results:
    if result.boxes.id is not None:
        ids = result.boxes.id.cpu().numpy()

        for id in ids:
            vehicle_ids.add(int(id))

print("Unique Vehicles:", len(vehicle_ids))

Unique Vehicles: 88


In [14]:
import cv2

cap = cv2.VideoCapture(video_path)

fps = cap.get(cv2.CAP_PROP_FPS)

print("FPS:", fps)

cap.release()

FPS: 23.976023976023978


In [15]:
vehicle_positions = {}

for frame_number, result in enumerate(results):

    if result.boxes.id is None:
        continue

    ids = result.boxes.id.cpu().numpy()
    boxes = result.boxes.xyxy.cpu().numpy()

    for id, box in zip(ids, boxes):

        x_center = (box[0]+box[2])/2
        y_center = (box[1]+box[3])/2

        if int(id) not in vehicle_positions:
            vehicle_positions[int(id)] = []

        vehicle_positions[int(id)].append(
            (frame_number, x_center, y_center)
        )


print("Tracked vehicles:", len(vehicle_positions))

Tracked vehicles: 88


In [16]:
import numpy as np

vehicle_speeds = {}

# Approximate conversion factor
# Need camera calibration for exact value
PIXEL_TO_METER = 0.05  

for vehicle_id, positions in vehicle_positions.items():

    if len(positions) < 2:
        continue

    distances = []

    for i in range(1, len(positions)):

        frame1, x1, y1 = positions[i-1]
        frame2, x2, y2 = positions[i]

        pixel_distance = np.sqrt(
            (x2-x1)**2 + (y2-y1)**2
        )

        distances.append(pixel_distance)


    total_pixel_distance = sum(distances)

    time_seconds = len(positions) / fps

    distance_meter = total_pixel_distance * PIXEL_TO_METER

    speed_mps = distance_meter / time_seconds

    speed_kmh = speed_mps * 3.6


    vehicle_speeds[vehicle_id] = speed_kmh


print("Speed calculated for vehicles:", len(vehicle_speeds))

Speed calculated for vehicles: 88


In [17]:
average_speed = np.mean(list(vehicle_speeds.values()))

print("Average Vehicle Speed:", round(average_speed,2), "km/h")

Average Vehicle Speed: 22.61 km/h


In [18]:
results = model.track(
    source=video_path,
    save=True,
    show=False,
    persist=True,
    classes=[2,3,5,7]
)


WARNING 
Inference results will accumulate in RAM unless `stream=True` is passed, which can cause out-of-memory errors for large
sources or long-running streams and videos. See https://docs.ultralytics.com/modes/predict/ for help.

Example:
    results = model(source=..., stream=True)  # generator of Results objects
    for r in results:
        boxes = r.boxes  # Boxes object for bbox outputs
        masks = r.masks  # Masks object for segment masks outputs
        probs = r.probs  # Class probabilities for classification outputs

video 1/1 (frame 1/660) C:\Users\DELL\OneDrive\Desktop\Final\Trafficvedio.mp4: 384x640 49 cars, 7 trucks, 130.7ms
video 1/1 (frame 2/660) C:\Users\DELL\OneDrive\Desktop\Final\Trafficvedio.mp4: 384x640 54 cars, 6 trucks, 73.0ms
video 1/1 (frame 3/660) C:\Users\DELL\OneDrive\Desktop\Final\Trafficvedio.mp4: 384x640 57 cars, 5 trucks, 71.3ms
video 1/1 (frame 4/660) C:\Users\DELL\OneDrive\Desktop\Final\Trafficvedio.mp4: 384x640 7 cars, 1 truck, 71.3ms
video 1/1 

In [19]:
import pickle

with open("vehicle_tracking_results.pkl", "wb") as f:
    pickle.dump(results, f)

print("YOLO tracking results saved")

YOLO tracking results saved


In [20]:
import pickle

with open("vehicle_tracking_results.pkl", "rb") as f:
    results = pickle.load(f)

print("YOLO results loaded")

YOLO results loaded


In [21]:
vehicle_positions = {}

for frame_number, result in enumerate(results):

    if result.boxes.id is None:
        continue

    ids = result.boxes.id.cpu().numpy()
    boxes = result.boxes.xyxy.cpu().numpy()

    for id, box in zip(ids, boxes):

        x_center = (box[0]+box[2])/2
        y_center = (box[1]+box[3])/2

        if int(id) not in vehicle_positions:
            vehicle_positions[int(id)] = []

        vehicle_positions[int(id)].append(
            (frame_number, x_center, y_center)
        )

print("Tracked vehicles:", len(vehicle_positions))

Tracked vehicles: 89


In [22]:
with open("vehicle_positions.pkl", "wb") as f:
    pickle.dump(vehicle_positions, f)

print("Vehicle positions saved")

Vehicle positions saved


In [23]:
vehicle_positions = pickle.load(open("vehicle_positions.pkl","rb"))